In [1]:
import pandas as pd
import numpy as np

food_atlas = pd.read_csv("Food Access Research Atlas.csv")

## Accuracy

In [14]:
print("ACCURACY")
# check for negative values in  numeric columns
numeric_cols = food_atlas.select_dtypes(include='number').columns
neg_report = {}

for col in numeric_cols:
    neg_mask = food_atlas[col] < 0
    cnt = neg_mask.sum()
    if cnt > 0:
        neg_report[col] = cnt

if neg_report:
    print("\n=== NEGATIVE VALUES FOUND ===")
    for col, cnt in neg_report.items():
        print(f"{col}: {cnt} negative values")
else:
    print("\nNo negative values found in any numeric column.")

ACCURACY

No negative values found in any numeric column.


## Completeness

In [10]:
print("MISSING VALUE OVERVIEW")
missing = food_atlas.isnull().sum()
missing_pct = (food_atlas.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Pct': missing_pct
})
# show cols with missing values
print(missing_report[missing_report['Missing_Count'] > 0])

#  identifying cols
crucial_cols = ['CensusTract', 'State', 'County', 'Pop2010']
for col in crucial_cols:
    if col in food_atlas.columns:
        if food_atlas[col].isnull().any():
            print(f"CRITICAL: {col} has missing values – {food_atlas[col].isnull().sum()} rows")
        else:
            print(f"{col}: complete")
    else:
        print(f"COLUMN {col} not found in dataset")

# empty rows
row_missing_all = food_atlas.isnull().all(axis=1).sum()
print(f"\nRows missing all values: {row_missing_all}")

MISSING VALUE OVERVIEW
                    Missing_Count  Missing_Pct
NUMGQTRS                       25         0.03
PCTGQTRS                       25         0.03
PovertyRate                     3         0.00
MedianFamilyIncome            748         1.03
LAPOP1_10                   29957        41.30
...                           ...          ...
TractAIAN                       4         0.01
TractOMultir                    4         0.01
TractHispanic                   4         0.01
TractHUNV                       4         0.01
TractSNAP                       4         0.01

[126 rows x 2 columns]
CensusTract: complete
State: complete
County: complete
Pop2010: complete

Rows missing all values: 0


## Consistency

In [13]:
print("CONSISTENCY")

# logic: low-access population must not exceed total population
pop_low_access_pairs = [
    ('Pop2010', 'LAPOP1_10'),
    ('Pop2010', 'TractLAPOP1'),
    ('Pop2010', 'TractLAPOP10'),
]
for pop_col, la_col in pop_low_access_pairs:
    if pop_col in food_atlas.columns and la_col in food_atlas.columns:
        mask = (food_atlas[la_col] > food_atlas[pop_col]) & food_atlas[pop_col].notna() & food_atlas[la_col].notna()
        violations = mask.sum()
        if violations > 0:
            print(f"CONSISTENCY: {la_col} > {pop_col} for {violations} tracts")
            print(food_atlas.loc[mask, [pop_col, la_col]].head())

# consistency: share fields should be within [0,100] (percentages) and possibly sum to 1 for related groups
share_cols = ['LAhalfshare', 'LA1and10share']
for col in share_cols:
    if col in food_atlas.columns:
        out_of_bounds = ((food_atlas[col] < 0) | (food_atlas[col] > 100)) & food_atlas[col].notna()
        print(f"{col}: {out_of_bounds.sum()} values outside [0,100]")

# flag consistency (LILATracts = low‑income AND low‑access)
# example, LILATracts_1And10 should be 1 only when both low‑income and low‑access flags are 1.
if 'LowIncomeTracts' in food_atlas.columns and 'LATracts1_10' in food_atlas.columns \
   and 'LILATracts_1And10' in food_atlas.columns:
    expected = ((food_atlas['LowIncomeTracts'] == 1) & (food_atlas['LATracts1_10'] == 1)).astype(int)
    mismatched = (food_atlas['LILATracts_1And10'] != expected)
    print(f"LILATracts_1And10 inconsistency: {mismatched.sum()} rows")

# consistency: urban/rural flag (Urban column) – should be 0 or 1
if 'Urban' in food_atlas.columns:
    valid = food_atlas['Urban'].isin([0,1])
    print(f"Urban flag invalid: {(~valid).sum()} rows")

# consistency: no duplicates on CensusTract
if 'CensusTract' in food_atlas.columns:
    dup_count = food_atlas['CensusTract'].duplicated().sum()
    print(f"\nDuplicate CensusTract entries: {dup_count}")
    if dup_count > 0:
        print(food_atlas[food_atlas['CensusTract'].duplicated(keep=False)].head())

CONSISTENCY
Urban flag invalid: 0 rows

Duplicate CensusTract entries: 0
